# Reproducing Saez et al. (2018) from CRCNS ofc-3

**Paper.** Saez I, Lin J, Stolk A, Chang E, Parvizi J, Schalk G, Knight RT, Hsu M (2018). Encoding of
multiple reward-related computations in transient and sustained high-frequency activity in human
OFC. *Current Biology* 28:2889-2899. [doi:10.1016/j.cub.2018.07.045](https://doi.org/10.1016/j.cub.2018.07.045).
The claims file (`claims/saez-2018.yaml`) was written from the version of record (main text,
legends, STAR Methods, Supplemental Information).

**Data set.** CRCNS ofc-3, "High-frequency activity of human orbitofrontal sites during
decision-making play", [doi:10.6080/K0VM49GF](https://doi.org/10.6080/K0VM49GF), read from the public
bucket `crcnsarchive` under `ofc-3/live/`. Version: `unversioned`. CRCNS does not version data
sets, and the paper cites the data set by its DOI only, so the files as published under
`live/` (data description version 0.7 of Feb 19, 2019, which added `ofc_behav_data.zip`) are the
only version that can be pinned. The MD5 sums of the files read, from the data set's
`checksums.md5`, are printed below so that a later change is detectable (data.zip
3531c772af4e4eb0c4e5332e74f94c81, ofc_behav_data.zip c35393e7734d33dc52e2c62b1a806274). The
folder `ofc-3/orig/` holds only older descriptions (no data).

**What the data set holds.** For 10 patients: window-averaged and 1-kHz HFA epochs around game
presentation and around a second event called "buttonpress" (the data show it is near the outcome
reveal, see the data check), the behaviour of every round with event times, the game of each round,
and the authors' regressors per HFA trial. It holds no broadband signal, no electrode locations
and no data of the 10 healthy participants.

| claim | tested here |
|---|---|
| methods-electrode-trial-counts | tested |
| behavior-risk-neutral | tested |
| fig1c-choice-win-probability | tested |
| figs2a-choice-gamble-ev | tested |
| fig1c-patients-like-controls | blocked, data_missing (healthy participants not deposited) |
| figs2b-rt-conflict | tested (patients only) |
| regressors-past-current-uncorrelated | tested |
| figs6a-regressor-collinearity | tested |
| fig3-present-encoding | tested |
| fig3-latency-duration-similar | tested |
| hfa-more-than-fp | blocked, data_missing (no broadband signal) |
| figs4b-power-at-buttonpress | blocked, data_missing (no broadband signal) |
| fig4-past-encoding | tested |
| fig5-past-sustained | tested |
| behavior-past-no-effect | tested |
| fig6-past-short-lived | tested |
| figs5d-sharp-boundary | tested |
| mixed-effects-robust | tested |
| figs6b-stepwise-similar | tested |
| figs3a-risk-all-trials | tested |
| figs3b-no-motor-encoding | tested |
| anatomy-brodmann-counts | blocked, data_missing (no electrode locations) |
| fig7-all-areas-similar | blocked, data_missing |
| no-anatomical-gradient | blocked, data_missing |
| tables2-patients-coverage | tested |
| fig7f-multiplexing | tested |
| figs7-present-past-overlap | tested |
| fig2-example-electrodes | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import multiprocessing
import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import saez

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")
started = time.time()

print("checksums.md5:", saez.checksums())
beh = saez.load_behaviour()
H = {}
for s in saez.SUBJECTS:
    r = beh[s]["rounds"]
    idx = saez.align_trials(r, beh[s]["globals"])
    beh[s]["idx"] = idx
    H[s] = saez.load_hfa(s, rt_ms=np.round(r["rt"].values[idx] * 1000))
    print(s, "HFA trials x windows x electrodes:", H[s]["game"].shape, f"{time.time() - started:.0f} s")

## Data check

In [ ]:
rows = []
for s in saez.SUBJECTS:
    r, g = beh[s]["rounds"], beh[s]["globals"]
    rows.append({"subject": s, "rounds": len(r), "timeouts": int(r.timeout.sum()),
                 "non_timeout": int((~r.timeout).sum()), "hfa_trials": H[s]["game"].shape[0],
                 "globals_rows": len(g), "electrodes": H[s]["game"].shape[2],
                 "gamble": int((r["choice.class"] == "Gamble").sum()),
                 "safebet": int((r["choice.class"] == "Safebet").sum())})
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
print("totals:", counts[["rounds", "timeouts", "non_timeout", "hfa_trials", "electrodes"]].sum().to_dict())
print("Description: 10 subjects, 200 rounds each, 192 electrodes; s10 has 138 rounds, and every subject",
      "but s08 and s09 has fewer HFA trials than non-timeout rounds.")

Each HFA trial is matched to a round (the files give no round number for HFA trials). The match uses
the authors' regressors in `sXX_behav_globals.mat` (win probability, gamble, win, regret, previous win
probability). The cell checks that every trial matched and that the other regressors agree with the
values computed from the csv files (Table S1 definitions), including the t-1 values.

In [ ]:
rows = []
for s in saez.SUBJECTS:
    r, g, idx = beh[s]["rounds"], beh[s]["globals"], beh[s]["idx"]
    p1 = saez.past_regressors(r, idx, 1)
    cur_mis = sum(int((~np.isclose(g[k].values, r[k].values[idx])).sum()) for k in saez.CURRENT)
    prev_mis = 0
    for k in saez.CURRENT:
        a, b = g["previous_" + k].values, p1[k].values
        prev_mis += int((~(np.isclose(a, b) | (np.isnan(a) & np.isnan(b)))).sum())
    dropped = sorted(set(np.where(~r.timeout.values)[0]) - set(idx))
    rows.append({"subject": s, "matched": len(idx), "mismatched current values": cur_mis,
                 "mismatched t-1 values": prev_mis, "non-timeout rounds without HFA": len(dropped),
                 "their round numbers": ",".join(str(d + 1) for d in dropped)})
print(pd.DataFrame(rows).to_string(index=False))
g = beh["s08"]["globals"]
print("\nValues of the authors' regressors (s08):")
for k in g.columns:
    print(f"  {k}: {np.unique(g[k].dropna()).round(3).tolist()[:12]}")
print("win probability = (10 - shown)/10 in every round:",
      all(np.allclose(beh[s]["rounds"].winprob, (10 - beh[s]["rounds"]["Risky.bet.shown.number"]) / 10) for s in saez.SUBJECTS))
print("risk = p(1-p) in the authors' file:", all(np.allclose(beh[s]["globals"].risk, beh[s]["globals"].winprob * (1 - beh[s]["globals"].winprob)) for s in saez.SUBJECTS))

The authors' regressor file settles three readings the claims file left open: win probability is
(10 - shown number)/10 (Table S1's reading, so a shown 2 gives 0.8), risk is p(1 - p) (reading (b) of
the claims file), and a t-1 regressor is the value of the previous round, with 0 for the choice and
outcome regressors when that round was a timeout (reading (a) would drop those trials; the file
keeps them with 0). The analysis uses the authors' values. The game is the same for every
subject (gamble_choices.csv has one row per round), and the outcomes in every csv agree with it:

In [ ]:
for s in saez.SUBJECTS:
    r = beh[s]["rounds"]
    ok = ~r.timeout
    q = r.rt[ok].quantile([0.5, 1.0]).values
    print(s, f"RT median {q[0]:.2f} s, max {q[1]:.2f} s, above 2 s: {int((r.rt[ok] > 2).sum())}", end=" | ")
    won_csv = r["outcome"].isin(["Win", "WouldHaveWon"])
    side = (r["choice.location"] == r["Risky.Side"]) == (r["choice.class"] == "Gamble")
    print(s, "outcome agrees with hidden > shown:", f"{(won_csv == r.gamble_won)[ok].mean():.3f}",
          "| chosen side agrees with Risky.Side:", f"{side[ok].mean():.3f}",
          "| ties shown = hidden:", int((r['Risky.bet.hidden.number'] == r['Risky.bet.shown.number']).sum()))
gc = beh["s01"]["rounds"]
print("\nprize values:", sorted(gc["Risky.Bet.Amount"].unique()), "| shown numbers:", sorted(gc["Risky.bet.shown.number"].unique()))
rt = pd.concat([beh[s]["rounds"].loc[~beh[s]["rounds"].timeout, "rt"] for s in saez.SUBJECTS])
lag_game = pd.concat([beh[s]["rounds"]["choice.time"] - beh[s]["rounds"]["newround.time"] for s in saez.SUBJECTS])
lag_rev = pd.concat([beh[s]["rounds"]["reveal.time"] - beh[s]["rounds"]["buttonpress.time"] for s in saez.SUBJECTS])
print(f"choice.time - newround.time (s): {lag_game.min():.3f} to {lag_game.max():.3f} (game presentation at 0.75 s per the paper)")
print(f"reveal.time - buttonpress.time (s): {lag_rev.min():.3f} to {lag_rev.max():.3f} (the paper: 0.55 s)")
print(f"reaction time buttonpress.time - choice.time (s), non-timeout: {rt.min():.3f} to {rt.max():.3f}, median {rt.median():.3f} (limit 2 s per the paper)")
print("non-timeout rounds with RT above 2 s:", int((rt > 2).sum()), "of", len(rt))

`choice.time`, a column the description does not list, is 0.66 to 0.97 s after the trial start: it is
read as game presentation. `reveal.time` is mostly 31 to 47 ms after the button press, not 550 ms as the
paper and the description say. Reaction times stay under 2 s in s01 to s05, reach 2.5 s in s06 to s09,
and in s10 have a median of 4.4 s and reach 80 s, with no timeouts: s10 does not follow the 2-s limit
the paper states (and played 138 rounds). The chosen side agrees with `Risky.Side` in every round of s01
to s06 but in only 87 to 97.5% of rounds of s07 to s10. The gamble prizes are $15 to $30; the paper
and the description say $10 to $30.

Next, the HFA arrays: window averages against the 1-kHz data, identical electrodes, value ranges.

In [ ]:
rows = []
for s in saez.SUBJECTS:
    h = H[s]
    rows.append({"subject": s, "electrodes": h["game"].shape[2],
                 "game raw range": tuple(round(v, 1) for v in h["game_raw_range"]),
                 "bp raw range": tuple(round(v, 1) for v in h["buttonpress_raw_range"]),
                 "NaN": h["game_nan"] + h["buttonpress_nan"],
                 "max |window - mean(raw)|": round(float(max(h["game_maxdiff"].max(), h["buttonpress_maxdiff"].max())), 4),
                 "electrodes whose windows differ": np.where((h["game_maxdiff"] > 1e-3) | (h["buttonpress_maxdiff"] > 1e-3))[0].tolist(),
                 "identical 1-kHz pairs": h["game_raw_identical"], "identical window pairs": h["game_win_identical"],
                 "window r > 0.95 pairs": h["game_high_corr"]})
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print("same identical pairs in the buttonpress arrays:",
      all(H[s]["game_raw_identical"] == H[s]["buttonpress_raw_identical"] and H[s]["game_win_identical"] == H[s]["buttonpress_win_identical"] for s in saez.SUBJECTS))

The window arrays are the mean of 201 samples (samples 50k to 50k + 200), so window k is centred
at -900 + 50k ms. In s03, electrodes 52 to 58 are exact copies of electrodes 45 to 51 in both the
1-kHz and the window arrays; electrode 42's 1-kHz trace is a copy of electrode 34's and its window
array a copy of electrode 41's (so its windows do not come from its own 1-kHz trace). With the
seven copied electrodes removed, s03 holds 52 electrodes and the data set 192, the counts of Table
S2. Electrode 42 holds no data of its own in either array. Following the protocol (each recording
counted once), the analysis drops every electrode identical to an earlier one in either array: 8 in
s03, leaving 191 electrodes. Pairs with r > 0.95 that are not identical (s05 58/60, s06 4/5) are kept.

In [ ]:
keep = {}
for s in saez.SUBJECTS:
    h = H[s]
    drop = sorted({j for i, j in h["game_raw_identical"] + h["game_win_identical"]})
    keep[s] = [e for e in range(h["game"].shape[2]) if e not in drop]
    for e in saez.EPOCHS:
        h[e] = h[e][:, :, keep[s]]
    if drop:
        print(s, "dropped electrodes", drop)
N_ELEC = sum(len(v) for v in keep.values())
both = sum(H[s]["game"].shape[2] for s in saez.SUBJECTS)
print("electrodes analysed:", N_ELEC, {s: len(v) for s, v in keep.items()})
wall = np.concatenate([H[s]["game"].ravel() for s in saez.SUBJECTS])
print(f"window HFA, game epoch: range {wall.min():.1f} to {wall.max():.1f}, mean {wall.mean():.3f}, SD {wall.std():.2f} (units not stated)")
print("SD over trials of the first window [-1000, -800] ms (mean over electrodes), against the window centred 500 ms:")
for s in saez.SUBJECTS:
    g, b = H[s]["game"].astype(float), H[s]["buttonpress"].astype(float)
    print(f"  {s}: game first window {g[:, 0].std(0).mean():.3f}, game at 500 ms {g[:, 28].std(0).mean():.2f}; buttonpress first window {b[:, 0].std(0).mean():.2f}")
del wall

The game epoch is baseline-corrected already: in every electrode its first window, [-1000, -800] ms,
is the same in every trial (SD 0.000 over trials, against about 4 to 6 later on). This is consistent with the paper's
200-ms baseline having been subtracted before deposit, with the baseline before the fixation cross
(game presentation is about 0.8 s after trial start). The buttonpress epoch is not baseline-corrected
in its own first window. The analysis uses the deposited values; the claims file's baseline readings
(a) and (b) are not applied, and reading (b) (the game baseline for both epochs) is the one consistent
with the data, as the next check shows that both epochs are cut from one trace.

**Clock.** The game and buttonpress epochs come from one continuous trace: for every trial checked,
the overlapping 1-kHz samples superimpose at one shift. The table gives that shift against the csv's
reaction time (button press minus `choice.time`).

In [ ]:
off = pd.concat([H[s]["offsets"].assign(subject=s) for s in saez.SUBJECTS])
print(off.groupby("subject").agg(trials=("trial", "size"), offset_minus_rt_min=("offset_minus_rt", "min"),
                                 offset_minus_rt_median=("offset_minus_rt", "median"),
                                 offset_minus_rt_max=("offset_minus_rt", "max"),
                                 sd_diff_median=("sd_diff", "median")).to_string())
print("trial-level HFA SD for comparison:", round(float(np.std(np.concatenate([H[s]['game'][:, 30].ravel() for s in saez.SUBJECTS]))), 2))

Per subject, the buttonpress epoch is centred a fixed 470 to 600 ms (the median per subject) after the
csv's button press, measured from game presentation (`choice.time`); the few trials far from the rest
have short overlaps. That is near the outcome reveal as the paper times it (550 ms after the press).
A contrast checks it: the gamble outcome (hidden number above the shown one, defined whether or not the
patient gambled) can be encoded only after the reveal. Below, the share of electrodes with
p < 0.05 for that variable in each window, against the 5% expected by chance.

In [ ]:
won = {s: beh[s]["rounds"].gamble_won.values[beh[s]["idx"]].astype(float) for s in saez.SUBJECTS}
sc_won = saez.scan_all(H, won, 0, np.random.default_rng(0))
c = saez.WIN_CENTRES_MS
share_won = sc_won["sig"].mean(2) * 100
print("window centre (ms):      ", c[::2].tolist())
print("game epoch, % p<0.05:    ", share_won[0, ::2].round(1).tolist())
print("buttonpress epoch, %:    ", share_won[1, ::2].round(1).tolist())
for e, i in (("game", 0), ("buttonpress", 1)):
    pre, post = share_won[i, (c >= -900) & (c <= -100)].mean(), share_won[i, (c >= 100) & (c <= 900)].mean()
    print(f"{e}: mean share, centres -900 to -100 ms {pre:.1f}%, 100 to 900 ms {post:.1f}%; first window centre >= 0 above 10%: "
          f"{c[(c >= 0) & (share_won[i] > 10)].min() if ((c >= 0) & (share_won[i] > 10)).any() else None}")

The gamble outcome is not encoded in the game epoch (near 5% throughout). In the buttonpress epoch
the share rises above chance in windows centred 200 to 400 ms, too early for a reveal 550 ms after
epoch time 0. The data are consistent with the "buttonpress" epoch being locked to the outcome
reveal, as Figure 3C labels it ("time locked to gamble outcome (t = 0)"), and not to the button press
as the data description says. This is consistent with the data, not shown: the true event times are
not in the files. In what follows, time 0 of the buttonpress epoch is read as outcome reveal (the
data's reading), and the claims file's reading (reveal at 550 ms) is shown alongside where timing
matters.

## methods-electrode-trial-counts

> Of 210 OFC electrodes recorded in 10 patients, 192 were artifact-free and included; each patient
> played 200 rounds, and timeouts, 9.98% of all trials, were excluded.

In [ ]:
table_s2_total = {"s01": 5, "s02": 6, "s03": 52, "s04": 5, "s05": 61, "s06": 7, "s07": 11, "s08": 10, "s09": 19, "s10": 16}  # paper, Table S2 last row
deposited = {s: len(keep[s]) + len({j for i, j in H[s]["game_raw_identical"] + H[s]["game_win_identical"]}) for s in saez.SUBJECTS}
copies_both = {s: len(set(H[s]["game_raw_identical"]) & set(H[s]["game_win_identical"])) for s in saez.SUBJECTS}   # same pair in both arrays
cmp = pd.DataFrame({"deposited": deposited, "without copies in both arrays": {s: deposited[s] - copies_both[s] for s in saez.SUBJECTS},
                    "unique recordings (analysed)": {s: len(keep[s]) for s in saez.SUBJECTS}, "Table S2": table_s2_total})
print(cmp.to_string()); print(cmp.sum().to_dict())
n_rounds, n_to = counts.rounds.sum(), counts.timeouts.sum()
print(f"patients: {len(saez.SUBJECTS)}")
print(f"rounds per patient: {counts.rounds.tolist()}")
print(f"timeouts {n_to} of {n_rounds} rounds = {100 * n_to / n_rounds:.2f}% (expected 9.98 +/- 0.998); of 2,000 = {100 * n_to / 2000:.2f}%")
missing = n_rounds - counts.hfa_trials.sum()
print(f"variant: rounds without an HFA trial (timeouts and other exclusions) {missing} of {n_rounds} = {100 * missing / n_rounds:.2f}%;"
      f" of 2,000 rounds {100 * (2000 - counts.hfa_trials.sum()) / 2000:.2f}%")

**Result: partially reproduced.** There are 10 patients. With the seven electrodes of s03 that are
copies of others in both arrays removed, the data set holds 192 electrodes, matching Table S2 patient
by patient (199 as deposited; 191 unique recordings, as s03 electrode 42 has no data of its own).
Timeouts are 99 of 1,938 rounds, 5.11%, outside 9.98 +/- 0.998, and s10 has 138 rounds, not 200. The
share of rounds without an HFA trial (timeouts and other exclusions), 9.55%, would fall inside the
tolerance, but the claims file defines the percentage on timeout trials.

## behavior-risk-neutral

> Patients were approximately risk neutral, choosing the gamble only slightly more often than a
> risk-neutral baseline model, on 55.9% (plus or minus 8.5%) of trials.

In [ ]:
B = pd.concat([beh[s]["rounds"].assign(subject=s) for s in saez.SUBJECTS], ignore_index=True)
B["ev"] = B.winprob * B["Risky.Bet.Amount"]
B["rn"] = np.where(B.ev > 10, 1.0, np.where(B.ev == 10, 0.5, 0.0))
Bn = B[~B.timeout].copy()
per = Bn.groupby("subject").agg(gamble_pct=("gamble_ind", lambda v: 100 * v.mean()), risk_neutral_pct=("rn", lambda v: 100 * v.mean()), n=("gamble_ind", "size"))
print(per.round(1).to_string())
print(f"(a) mean over patients {per.gamble_pct.mean():.1f}% (SD {per.gamble_pct.std():.1f}, SEM {per.gamble_pct.sem():.1f}); risk-neutral model {per.risk_neutral_pct.mean():.1f}%")
print(f"(b) pooled {100 * Bn.gamble_ind.mean():.1f}%; risk-neutral model {100 * Bn.rn.mean():.1f}%")
print("expected 55.9 +/- 8.5 (47.4 to 64.4)")

**Result: reproduced.** Patients chose the gamble on 54.9% of non-timeout rounds averaged over
patients (SD 8.6) and 55.1% pooled, within 55.9 +/- 8.5 under both readings; a risk-neutral chooser
(gamble when p times prize exceeds $10) would gamble on 53.0%.

## fig1c-choice-win-probability

> The proportion of risky choices increased with win probability, well captured by a logit model
> (p < 0.001, random effects logit analysis).

In [ ]:
print("proportion of gamble choices by win probability (mean over patients):")
print((Bn.groupby(["winprob", "subject"]).gamble_ind.mean().groupby("winprob").mean() * 100).round(1).to_string())
X = np.c_[np.ones(len(Bn)), Bn.winprob.values]
fit_a = saez.glmm_logit(Bn.gamble_ind.values, X, Bn.subject.values)
fit_b = saez.glmm_logit(Bn.gamble_ind.values, X, Bn.subject.values, random_slope_col=1)
for name, f in (("(a) random intercept", fit_a), ("(b) random intercept and slope", fit_b)):
    print(f"{name}: slope {f.estimate[1]:.2f} (SE {f.se[1]:.2f}), z {f.z[1]:.1f}, p {f.p[1]:.2e}; RE SD {np.round(f.attrs['re_sd'], 2).tolist()}, converged {f.attrs['converged']}")
print("per patient logit slopes (fixed-effect fits, for the breakdown):")
for s, d in Bn.groupby("subject"):
    f = saez.glmm_logit(d.gamble_ind.values, np.c_[np.ones(len(d)), d.winprob.values], np.zeros(len(d)))
    print(f"  {s}: slope {f.estimate[1]:.1f}, p {f.p[1]:.1e}")

**Result: reproduced.** The slope of gamble choice on win probability is positive with p < 1e-60
under both random-effects readings (4.79 and 5.75 per unit probability). Per patient the slope is
positive and significant in 8 of 10; s04 (slope -0.6) and s10 (0.7) show no significant effect.

## figs2a-choice-gamble-ev

> Patients' choices were sensitive to the expected value of the gamble (prize times win
> probability), with gamble choices more frequent at higher gamble expected value.

In [ ]:
print("proportion of gamble choices by gamble expected value (mean over patients, % ; n rounds):")
tab = Bn.groupby(["ev", "subject"]).gamble_ind.mean().groupby("ev").mean() * 100
print(pd.DataFrame({"gamble %": tab.round(1), "rounds": Bn.groupby("ev").size()}).T.to_string())
X = np.c_[np.ones(len(Bn)), Bn.ev.values]
for name, f in (("(a) random intercept", saez.glmm_logit(Bn.gamble_ind.values, X, Bn.subject.values)),
                ("(b) random intercept and slope", saez.glmm_logit(Bn.gamble_ind.values, X, Bn.subject.values, random_slope_col=1))):
    print(f"{name}: slope per $ {f.estimate[1]:.3f} (SE {f.se[1]:.3f}), p {f.p[1]:.2e}")

**Result: reproduced.** Gamble choices rise from 16.9% at an expected value of $0 to 68 to 87% above
$15, and the random-effects logit slope is positive (0.16 per $) with p < 1e-40 under both readings.

## figs2b-rt-conflict

> Reaction times were longer in high-conflict trials (win probability 40 to 60%) than in
> low-conflict trials (below 30% or above 70%), in patients and in healthy subjects (both p < 0.001,
> t test), with comparable effect sizes.

RT is `buttonpress.time` minus `choice.time` (game presentation, see the data check). Only the
patients are in the data set.

In [ ]:
Bn["conflict"] = np.where(Bn.winprob.between(0.4, 0.6), "high", np.where((Bn.winprob < 0.3) | (Bn.winprob > 0.7), "low", "mid"))
print(Bn.groupby("conflict").rt.agg(["size", "mean", "std"]).round(3).to_string())
hi, lo = Bn.rt[Bn.conflict == "high"], Bn.rt[Bn.conflict == "low"]
t_a = stats.ttest_ind(hi, lo)
d_a = (hi.mean() - lo.mean()) / np.sqrt(((len(hi) - 1) * hi.var() + (len(lo) - 1) * lo.var()) / (len(hi) + len(lo) - 2))
m = Bn[Bn.conflict != "mid"].groupby(["subject", "conflict"]).rt.mean().unstack()
t_b = stats.ttest_rel(m["high"], m["low"])
diff = m["high"] - m["low"]
print(f"(a) pooled trials: high {hi.mean():.3f} s vs low {lo.mean():.3f} s, t = {t_a.statistic:.2f}, p = {t_a.pvalue:.2e}, Cohen's d = {d_a:.2f}")
print(f"(b) paired over patients: mean difference {diff.mean() * 1000:.0f} ms, t = {t_b.statistic:.2f}, p = {t_b.pvalue:.2e}, d_z = {diff.mean() / diff.std():.2f}")
print("per patient high - low (ms):", (diff * 1000).round(0).to_dict())
v = Bn[(Bn.subject != "s10") & (Bn.conflict != "mid")]
tv = stats.ttest_ind(v.rt[v.conflict == "high"], v.rt[v.conflict == "low"])
mv = m.drop("s10")
print(f"variant without s10 (RTs not bounded, see the data check): pooled t = {tv.statistic:.2f}, p = {tv.pvalue:.2e};"
      f" paired over 9 patients p = {stats.ttest_rel(mv['high'], mv['low']).pvalue:.2e}")

**Result: not reproduced.** High-conflict RTs are longer on average (1.691 against 1.476 s pooled; 179
ms per patient), but neither reading reaches p < 0.001: p = 0.165 over pooled trials (Cohen's d 0.07)
and p = 0.104 over the 10 patients. s10's unbounded RTs (up to 80 s) dominate the pooled test; without
s10 the pooled test gives p = 2.2e-06 and the paired one p = 0.147. The healthy subjects' part
cannot be tested.

## Regressors per HFA trial

Current and t-1 regressors are the authors' (`sXX_behav_globals.mat`); t-2 to t-5 are built from the
rounds with the same convention (helper, `past_regressors`).

In [ ]:
LAGS = range(6)
REG = {}   # (regressor, lag) -> subject -> values per HFA trial
for s in saez.SUBJECTS:
    g, r, idx = beh[s]["globals"], beh[s]["rounds"], beh[s]["idx"]
    for k in saez.CURRENT:
        REG.setdefault((k, 0), {})[s] = g[k].values
        REG.setdefault((k, 1), {})[s] = g["previous_" + k].values
    for lag in range(2, 6):
        p = saez.past_regressors(r, idx, lag)
        for k in saez.CURRENT:
            REG.setdefault((k, lag), {})[s] = p[k].values
    REG.setdefault(("left", 0), {})[s] = r["left"].values[idx]
    REG.setdefault(("risk_gamble_only", 0), {})[s] = np.where(g.gamble_ind.values == 1, g.risk.values, np.nan)
print(len(REG), "regressors;", "NaN trials per lag (all subjects):",
      {lag: int(sum(np.isnan(REG[("winprob", lag)][s]).sum() for s in saez.SUBJECTS)) for lag in LAGS})

## regressors-past-current-uncorrelated

> Past and current regressors were nearly uncorrelated (mean R2 2.1%, maximum R2 3.7%).

In [ ]:
def r2_pairs(pairs):
    """R2 per pair: mean over patients of within-patient R2, and R2 on pooled trials."""
    per_pat, pooled = [], []
    for a, b in pairs:
        vals = []
        xa, xb = [], []
        for s in saez.SUBJECTS:
            u, v = REG[a][s], REG[b][s]
            ok = ~np.isnan(u) & ~np.isnan(v)
            if u[ok].std() > 0 and v[ok].std() > 0:
                vals.append(np.corrcoef(u[ok], v[ok])[0, 1] ** 2)
            xa.append(u[ok]); xb.append(v[ok])
        per_pat.append(np.mean(vals))
        pooled.append(np.corrcoef(np.concatenate(xa), np.concatenate(xb))[0, 1] ** 2)
    return np.array(per_pat) * 100, np.array(pooled) * 100

readings = {"(a) own t-1 (8 pairs)": [((k, 0), (k, 1)) for k in saez.CURRENT],
            "(b) all current x t-1 (64)": [((k, 0), (j, 1)) for k in saez.CURRENT for j in saez.CURRENT],
            "(c) all current x t-1..t-5 (320)": [((k, 0), (j, lag)) for k in saez.CURRENT for j in saez.CURRENT for lag in range(1, 6)]}
for name, pairs in readings.items():
    pp, pl = r2_pairs(pairs)
    print(f"{name}: averaged over patients mean {pp.mean():.2f}% max {pp.max():.2f}% | pooled trials mean {pl.mean():.2f}% max {pl.max():.2f}%")
print("expected mean 2.1 +/- 0.21, max 3.7 +/- 0.37")

**Result: not reproduced.** Past and current regressors are even less correlated than the paper
reports: the mean R2 is 0.21 to 0.59% and the maximum 0.64 to 2.29% across the readings, so no reading
is within 2.1 +/- 0.21% and 3.7 +/- 0.37%. The finding that they are nearly uncorrelated holds.

## figs6a-regressor-collinearity

> Regressors were weakly correlated between and within categories (mean R2 0.15, all R2 below 0.6).

In [ ]:
cur = [(k, 0) for k in saez.CURRENT]
both16 = cur + [(k, 1) for k in saez.CURRENT]
for name, regs in (("(a) 8 current", cur), ("(b) 8 current + 8 t-1", both16)):
    pairs = [(regs[i], regs[j]) for i in range(len(regs)) for j in range(i + 1, len(regs))]
    pp, pl = r2_pairs(pairs)
    print(f"{name}, {len(pairs)} pairs: averaged over patients mean {pp.mean() / 100:.4f} max {pp.max() / 100:.3f} | pooled mean {pl.mean() / 100:.4f} max {pl.max() / 100:.3f}")
    top = sorted(zip(pl / 100, pairs), reverse=True)[:4]
    print("   largest pooled R2:", [(round(v, 2), a[0], a[1], b[0], b[1]) for v, (a, b) in top])
print("expected mean 0.15 +/- 0.015, all below 0.6")

**Result: reproduced.** For the 8 current regressors (reading (a)) the mean R2 over the 28 pairs is
0.155 on pooled trials and 0.165 averaged over patients, both within 0.15 +/- 0.015, and no pair
reaches 0.6 (largest 0.55 pooled, 0.57 averaged, loss index with RPE). With the t-1 regressors added (reading (b)) the mean falls to
0.07 to 0.08, outside the tolerance.

## Encoding scans

For each regressor, lag and electrode: a linear regression of window HFA on the regressor in each of
the 57 windows of both epochs; an electrode encodes the regressor when p < 0.05 in 5 or more
consecutive windows of either epoch. Chance: 1,000 shuffles of the regressor across trials (the same
shuffle for both epochs), each pooled over subjects.

In [ ]:
t0 = time.time()
SC = {}
for i, key in enumerate(REG):
    SC[key] = saez.scan_all(H, REG[key], 1000, np.random.default_rng(1000 + i))
print(f"{len(SC)} scans in {time.time() - t0:.0f} s")
SUBJ = SC[("risk", 0)]["subject"]

def summary(key):
    sc = SC[key]
    return {"regressor": saez.LABELS.get(key[0], key[0]), "lag": key[1], "encoding": int(sc["encoding"].sum()),
            "percent": round(100 * sc["encoding"].mean(), 1), "chance mean": round(sc["perm_count"].mean(), 1),
            "chance 95th": float(np.percentile(sc["perm_count"], 95)), "patients": len(set(SUBJ[sc["encoding"]]))}

## fig3-present-encoding

> OFC HFA encodes the choice regressors (risk, gamble choice, offer value, chosen value), time locked
> to game presentation, and the outcome regressors (gamble win, gamble loss, RPE, regret), time locked
> to outcome reveal, with average %EV above chance and a rise in the number of encoding electrodes.

In [ ]:
expected_cur = {"risk": 50, "gamble_ind": 34, "winprob": 31, "exputil": 26, "win_ind": 45, "loss_ind": 33, "rpe": 28, "regret": 35}
t3 = pd.DataFrame([summary((k, 0)) for k in saez.CURRENT])
t3["expected"] = [expected_cur[k] for k in saez.CURRENT]
t3["within 10%"] = (t3.encoding - t3.expected).abs() <= 0.1 * t3.expected
t3["above chance 95th"] = t3.encoding > t3["chance 95th"]
t3["own-null variant"] = [int((SC[(k, 0)]["encoding"] & (SC[(k, 0)]["run"].max(0) > SC[(k, 0)]["run95"])).sum()) for k in saez.CURRENT]
print(t3.to_string(index=False))
print(f"false-positive rate of the selection under shuffles: {np.mean([SC[(k, 0)]['perm_count'].mean() for k in saez.CURRENT]) / N_ELEC * 100:.1f}% of electrodes")
print("\nper patient (data / Table S2):")
table_s2 = {"risk": [1, 1, 25, 0, 14, 0, 0, 3, 4, 2], "gamble_ind": [2, 1, 7, 0, 14, 0, 1, 4, 3, 2], "winprob": [0, 0, 6, 1, 9, 0, 1, 6, 7, 1],
            "exputil": [0, 0, 5, 2, 8, 0, 3, 1, 6, 1], "win_ind": [0, 1, 7, 0, 16, 0, 0, 5, 13, 3], "loss_ind": [3, 1, 2, 0, 14, 0, 3, 1, 8, 1],
            "rpe": [0, 1, 2, 0, 10, 0, 2, 1, 11, 1], "regret": [0, 3, 4, 1, 14, 0, 1, 2, 6, 4]}   # paper, Table S2
for k in saez.CURRENT:
    e = SC[(k, 0)]["encoding"]
    print(f"  {saez.LABELS[k]:24s}", " ".join(f"{int(e[SUBJ == s].sum()):2d}/{table_s2[k][i]:<2d}" for i, s in enumerate(saez.SUBJECTS)))


def profile(keys, epoch):
    """Mean %EV over each regressor's encoding electrodes, averaged over the regressors; chance profile."""
    ei = saez.EPOCHS.index(epoch)
    prof = np.mean([SC[k]["r2"][ei][:, SC[k]["encoding"]].mean(1) * 100 for k in keys], 0)
    ch = np.mean([SC[k]["perm_r2_enc"][:, ei].sum(0) / SC[k]["perm_count"].sum() * 100 for k in keys], 0)
    return prof, ch


def share(keys, epoch, among_encoding):
    ei = saez.EPOCHS.index(epoch)
    if among_encoding:
        return np.mean([SC[k]["sig"][ei][:, SC[k]["encoding"]].mean(1) * 100 for k in keys], 0)
    return np.mean([SC[k]["sig"][ei].mean(1) * 100 for k in keys], 0)


c = saez.WIN_CENTRES_MS
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (name, keys, epoch, onset) in zip(axes, (("choice", [(k, 0) for k in saez.CHOICE], "game", 0),
                                                ("outcome", [(k, 0) for k in saez.OUTCOME], "buttonpress", 0))):
    prof, ch = profile(keys, epoch)
    sa, sb = share(keys, epoch, True), share(keys, epoch, False)
    pk = c[prof.argmax()]
    print(f"\n{name} regressors, {epoch} epoch: peak mean %EV {prof.max():.2f} at window centre {pk} ms (chance profile {ch.min():.2f} to {ch.max():.2f});"
          f" windows above chance: {int((prof > ch).sum())} of 57")
    print(f"   share among encoding electrodes (a) peaks at {c[sa.argmax()]} ms ({sa.max():.1f}%); share of all electrodes (b) peaks at {c[sb.argmax()]} ms ({sb.max():.1f}%)")
    if name == "outcome":
        print(f"   reveal at epoch time 0 (data): peak {pk} ms after reveal; reveal at +550 ms (claims file): peak {pk - 550} ms after reveal")
    ax.plot(c, prof, label="mean %EV"); ax.plot(c, ch, "k:", label="chance")
    ax2 = ax.twinx(); ax2.bar(c, sb, width=40, alpha=0.3, color="gray"); ax2.set_ylabel("% electrodes p<0.05")
    ax.set_title(f"{name}, {epoch} epoch"); ax.set_xlabel("window centre (ms)"); ax.set_ylabel("%EV"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Result: not reproduced.** The timing holds: the mean %EV of the choice regressors peaks 750 ms after
game presentation and that of the outcome regressors 800 ms after time 0 of the buttonpress epoch
(250 ms after a reveal at +550 ms, the claims file's timing), with the per-window share of encoding
electrodes peaking in the same post-event period (850 ms under both readings, and 650 ms for outcome
under reading (a)). The counts do not: none of the eight is within 10% of Table S2 (41 to 84 against
26 to 50), and chosen value (41 electrodes) does not exceed the 95th percentile of its shuffles (50).
The selection rule itself classifies 18.2% of electrodes as encoding under shuffled labels (about 35
of 191), more than Table S2 reports for chosen value (26). If an electrode is also required to beat
the 95th percentile of its own shuffled run lengths (a variant, not the claims file's reading), the
counts are 12 to 41, mostly below Table S2. Per patient, s03 and s05 hold most encoding electrodes in
both the data and Table S2.

## fig3-latency-duration-similar

> The delays of choice encoding after game presentation and of outcome encoding after outcome reveal
> did not differ (512.5 plus or minus 59 ms against 287 plus or minus 114.33 ms, p = 0.14, t test), nor
> did their transient durations (612.15 plus or minus 132.9 ms against 712.5 plus or minus 112.5 ms,
> p = 0.58).

In [ ]:
def run_around(mask, k):
    """Length of the run of True in mask that contains index k."""
    if not mask[k]:
        return 0
    a = k
    while a > 0 and mask[a - 1]:
        a -= 1
    b = k
    while b < len(mask) - 1 and mask[b + 1]:
        b += 1
    return b - a + 1


def latency_table(reveal_ms):
    rows = []
    for k in saez.CURRENT:
        epoch, event = ("game", 0) if k in saez.CHOICE else ("buttonpress", reveal_ms)
        ei = saez.EPOCHS.index(epoch)
        sc = SC[(k, 0)]
        prof = sc["r2"][ei][:, sc["encoding"]].mean(1) * 100
        ch = sc["perm_r2_enc"][:, ei].sum(0) / sc["perm_count"].sum() * 100
        after = c - 100 >= event            # windows that start at or after the event
        kpk = np.where(after)[0][prof[after].argmax()]
        dur = run_around(prof > ch, kpk) * 50
        shr = sc["sig"][ei].mean(1)
        thr = np.percentile(sc["perm_sig_count"][:, ei] / N_ELEC, 95, axis=0)
        above = np.where(after & (shr > thr))[0]
        rows.append({"regressor": saez.LABELS[k], "group": "choice" if k in saez.CHOICE else "outcome",
                     "(a) delay": c[kpk] - event, "(a,b) duration": dur,
                     "(b) delay": (c[above[0]] - event) if len(above) else np.nan})
    return pd.DataFrame(rows)


def per_electrode(reveal_ms):
    rows = []
    for k in saez.CURRENT:
        epoch, event = ("game", 0) if k in saez.CHOICE else ("buttonpress", reveal_ms)
        ei = saez.EPOCHS.index(epoch)
        sc = SC[(k, 0)]
        for e in np.where(sc["run"][ei] >= saez.MIN_RUN)[0]:
            st, ln = saez.run_bounds(sc["sig"][ei][:, e])
            rows.append({"group": "choice" if k in saez.CHOICE else "outcome", "delay": c[st] - event, "duration": ln * 50})
    return pd.DataFrame(rows)


def ttest_line(a, b, label):
    t = stats.ttest_ind(a, b, nan_policy="omit")
    return (f"{label}: choice {np.nanmean(a):.1f} +/- {stats.sem(a, nan_policy='omit'):.1f} vs outcome {np.nanmean(b):.1f} +/- "
            f"{stats.sem(b, nan_policy='omit'):.1f} ms, t = {t.statistic:.2f}, p = {t.pvalue:.3f}")


for reveal, name in ((0, "reveal at buttonpress-epoch 0 (data)"), (550, "reveal at +550 ms (claims file)")):
    lt = latency_table(reveal)
    print(f"\n{name}\n", lt.to_string(index=False))
    ch, oc = lt[lt.group == "choice"], lt[lt.group == "outcome"]
    print(ttest_line(ch["(a) delay"], oc["(a) delay"], "(a) delay"))
    print(ttest_line(ch["(a,b) duration"], oc["(a,b) duration"], "(a,b) duration"))
    print(ttest_line(ch["(b) delay"], oc["(b) delay"], "(b) delay"))
    pe = per_electrode(reveal)
    print(ttest_line(pe.delay[pe.group == "choice"], pe.delay[pe.group == "outcome"], f"(c) delay, {len(pe)} electrode-regressor pairs"))
    print(ttest_line(pe.duration[pe.group == "choice"], pe.duration[pe.group == "outcome"], "(c) duration"))
print("expected: delay 512.5 +/- 59 vs 287 +/- 114.33 ms (p = 0.14); duration 612.15 +/- 132.9 vs 712.5 +/- 112.5 ms (p = 0.58)")

**Result: reproduced, resting on the reading of the buttonpress epoch as locked to the outcome reveal.**
With the reveal at time 0 of the buttonpress epoch (the data's reading), the choice and outcome delays
do not differ under reading (a) (800 +/- 74 against 650 +/- 87 ms, p = 0.235) or reading (b) (512.5
+/- 101 against 325 +/- 132 ms, p = 0.301), nor do the durations (687.5 +/- 155 against 1,000 +/-
158 ms, p = 0.207). Under reading (b) both delays are within the paper's values (512.5 +/- 59 and 287
+/- 114.33 ms) and the choice duration is within 612.15 +/- 132.9 ms; the outcome duration is not
(1,000 against 712.5 +/- 112.5). With the reveal at +550 ms (the claims file's timing) outcome encoding
starts earlier than choice encoding (p = 0.006 under (b)), and the per-electrode reading (c), less
supported, gives p = 0.026 for the delays under the data's timing.

## fig4-past-encoding

> HFA encodes characteristics of the preceding trial; past choice and outcome features are widely
> represented across OFC except past risk, dissociable from current signals.

In [ ]:
expected_past = {"risk": 17, "gamble_ind": 43, "winprob": 46, "exputil": 35, "win_ind": 49, "loss_ind": 56, "rpe": 50, "regret": 64}
t4 = pd.DataFrame([summary((k, 1)) for k in saez.CURRENT])
t4["expected"] = [expected_past[k] for k in saez.CURRENT]
t4["within 10%"] = (t4.encoding - t4.expected).abs() <= 0.1 * t4.expected
t4["above chance 95th"] = t4.encoding > t4["chance 95th"]
t4["own-null variant"] = [int((SC[(k, 1)]["encoding"] & (SC[(k, 1)]["run"].max(0) > SC[(k, 1)]["run95"])).sum()) for k in saez.CURRENT]
print(t4.to_string(index=False))
print(f"mean proportion of encoding electrodes over the 16 current and t-1 regressors: {(t3.percent.sum() + t4.percent.sum()) / 16:.1f}% (paper, HFA: 20.9%)")
print("past risk has the lowest count:", t4.encoding.iloc[0] == t4.encoding.min(), "| rank of past risk (1 = lowest):", int(t4.encoding.rank(method="min").iloc[0]))
for ep in saez.EPOCHS:
    prof, ch = profile([(k, 1) for k in saez.CURRENT], ep)
    print(f"t-1 regressors, {ep} epoch: peak mean %EV {prof.max():.2f} at {c[prof.argmax()]} ms; windows above chance {int((prof > ch).sum())} of 57")

**Result: partially reproduced.** Past features are encoded: the seven t-1 regressors other than risk
each exceed the 95th percentile of their shuffles (53 to 95 electrodes against 48 to 50), and past risk
has the lowest count (40, below its 95th percentile of 49). None of the eight counts is within 10% of
Table S2 (40 to 95 against 17 to 64). The mean %EV of the t-1 regressors is above its chance profile in
56 of 57 game-epoch windows and all 57 buttonpress-epoch windows.

## fig5-past-sustained

> Past-trial signals were sustained; the longest continuous stretch of significant encoding was longer
> for past outcome (1,341.37 plus or minus 49.63 ms) and past choice (1,169.15 plus or minus 61.2 ms)
> than for current choice (799.6 plus or minus 36.94 ms) and current outcome (706.9 plus or minus
> 30.56 ms), all p < 10^-6, t test.

In [ ]:
GROUPS = {"choice": [(k, 0) for k in saez.CHOICE], "outcome": [(k, 0) for k in saez.OUTCOME],
          "past choice": [(k, 1) for k in saez.CHOICE], "past outcome": [(k, 1) for k in saez.OUTCOME]}
exp5 = {"past outcome": 1341.37, "past choice": 1169.15, "choice": 799.6, "outcome": 706.9}
sem5 = {"past outcome": 49.63, "past choice": 61.2, "choice": 36.94, "outcome": 30.56}


def durations(epoch_reading):
    out = {}
    for gname, keys in GROUPS.items():
        vals = []
        for k in keys:
            sc = SC[k]
            if epoch_reading == "a":
                n = sc["run"].max(0)
            else:
                n = sc["run"][1 if gname == "outcome" else 0]
            vals.append(n[n >= saez.MIN_RUN])
        out[gname] = np.concatenate(vals).astype(float)
    return out


for er, ename in (("a", "longest over both epochs"), ("b", "choice and past in game epoch, outcome in buttonpress epoch")):
    d = durations(er)
    for lr, f in (("(a) windows x 50 ms", lambda n: n * 50), ("(b)/(c) windows x 50 + 150 ms", lambda n: n * 50 + 150)):
        print(f"\nepoch reading ({er}) {ename}; length {lr}")
        for gname in GROUPS:
            v = f(d[gname])
            print(f"   {gname:13s} n = {len(v):3d}  mean {v.mean():7.1f} +/- {stats.sem(v):5.1f} ms  (expected {exp5[gname]} +/- {sem5[gname]};"
                  f" within SEM {abs(v.mean() - exp5[gname]) <= sem5[gname]}, within 10% {abs(v.mean() - exp5[gname]) <= 0.1 * exp5[gname]})")
        for p in ("past outcome", "past choice"):
            for q in ("choice", "outcome"):
                t = stats.ttest_ind(f(d[p]), f(d[q]))
                print(f"   {p} vs {q}: t = {t.statistic:.2f}, p = {t.pvalue:.1e}")

**Result: partially reproduced.** Past signals last longer: under epoch reading (a) both past groups
have longer stretches than both current groups in all four comparisons with p < 0.01 (p = 8.0e-07 to
8.8e-03), but only past outcome against current outcome reaches p < 10^-6. Under epoch reading (b) past
choice against current choice gives p = 0.011. The durations are much shorter than the paper's under
every reading (for example 820 and 801 ms for past outcome and past choice against 1,341 and 1,169, and
684 and 612 ms for current choice and outcome against 800 and 707, with the 150 ms of the window added).

## fig6-past-short-lived

> Encoding of past information was limited to the immediately preceding trial; the proportion of
> electrodes encoding choices and outcomes 2 to 5 trials back was significantly lower than for the
> current and previous trials (p < 0.01, t test), with no evidence of encoding from earlier rounds.

In [ ]:
prop = pd.DataFrame({lag: {saez.LABELS[k]: 100 * SC[(k, lag)]["encoding"].mean() for k in saez.CURRENT} for lag in LAGS})
prop.columns = [f"t-{lag}" if lag else "t" for lag in LAGS]
print("percent of electrodes encoding, per regressor and lag:\n", prop.round(1).to_string())
chance = {lag: np.mean([SC[(k, lag)]["perm_count"] for k in saez.CURRENT], 0) / N_ELEC * 100 for lag in LAGS}
lagmean = prop.mean()
print("\nmean over regressors per lag:", lagmean.round(1).to_dict())
print("chance (mean over regressors) per lag, mean and 95th percentile:",
      {f"t-{lag}": (round(chance[lag].mean(), 1), round(np.percentile(chance[lag], 95), 1)) for lag in LAGS})
far = prop[[f"t-{lag}" for lag in range(2, 6)]].values.ravel()
ta, tb = stats.ttest_ind(far, prop["t"]), stats.ttest_ind(far, prop["t-1"])
print(f"(a) per-regressor proportions, t-2..t-5 pooled (mean {far.mean():.1f}%) vs t ({prop['t'].mean():.1f}%): t = {ta.statistic:.2f}, p = {ta.pvalue:.2e};"
      f" vs t-1 ({prop['t-1'].mean():.1f}%): t = {tb.statistic:.2f}, p = {tb.pvalue:.2e}")
lm = lagmean.values[2:]
ra, rb = stats.ttest_1samp(lm, lagmean["t"]), stats.ttest_1samp(lm, lagmean["t-1"])
print(f"(b) per-lag means t-2..t-5 {lm.round(1).tolist()} vs t {lagmean['t']:.1f}: p = {ra.pvalue:.2e}; vs t-1 {lagmean['t-1']:.1f}: p = {rb.pvalue:.2e}")
print("lags t-2..t-5 whose mean proportion exceeds the 95th percentile of chance:",
      [f"t-{lag}" for lag in range(2, 6) if lagmean.iloc[lag] > np.percentile(chance[lag], 95)])
above = {f"t-{lag}": [saez.LABELS[k] for k in saez.CURRENT if SC[(k, lag)]["encoding"].sum() > np.percentile(SC[(k, lag)]["perm_count"], 95)] for lag in range(2, 6)}
print("regressors above their own 95th percentile at t-2..t-5:", above)

**Result: reproduced.** The proportion of encoding electrodes falls from 31.8% (t) and 34.9% (t-1) to
16.6 to 19.4% at t-2 to t-5 (p = 2.9e-10 and 7.6e-11 under reading (a); 1.9e-04 and 1.1e-04 under
reading (b)). At no lag from t-2 to t-5 does the mean proportion exceed the 95th percentile of chance
(20.5 to 20.7%); only loss index at t-3 exceeds its own 95th percentile.

## figs5d-sharp-boundary

> The distribution of encoding significance at the peak %EV shows a sharp boundary between t-1 and
> t-2 but not between t and t-1 (one-sided Kolmogorov-Smirnov tests).

Per electrode, the window with the highest %EV over both epochs, and -log10 p of the regression
there. "t-1>t" is tested as the ECDF of t-1 lying above that of t (`ks_2samp(t-1, t,
alternative="greater")`), and "t-2>t-1" likewise.

In [ ]:
def peak_logp(key):
    sc = SC[key]
    r2 = sc["r2"].reshape(-1, sc["r2"].shape[2])
    best = r2.max(0)
    n = np.array([sc["n"][s] for s in SUBJ])
    t = np.sqrt(best * (n - 2) / np.maximum(1 - best, 1e-12))
    return -np.log10(2 * stats.t.sf(t, n - 2))


rows = []
pooled = {lag: [] for lag in LAGS}
for k in saez.CURRENT:
    lp = {lag: peak_logp((k, lag)) for lag in LAGS}
    for lag in LAGS:
        pooled[lag].append(lp[lag])
    a = stats.ks_2samp(lp[1], lp[0], alternative="greater")
    b = stats.ks_2samp(lp[2], lp[1], alternative="greater")
    rows.append({"regressor": saez.LABELS[k], "median -log10p t": np.median(lp[0]), "t-1": np.median(lp[1]), "t-2": np.median(lp[2]),
                 "KS t-1>t p": a.pvalue, "KS t-2>t-1 p": b.pvalue})
ks = pd.DataFrame(rows)
print(ks.round(4).to_string(index=False))
P = {lag: np.concatenate(pooled[lag]) for lag in LAGS}
a, b = stats.ks_2samp(P[1], P[0], alternative="greater"), stats.ks_2samp(P[2], P[1], alternative="greater")
print(f"pooled: t-1>t D = {a.statistic:.3f}, p = {a.pvalue:.3g}; t-2>t-1 D = {b.statistic:.3f}, p = {b.pvalue:.3g}")
n_ok = int(((ks["KS t-2>t-1 p"] < 0.05) & (ks["KS t-1>t p"] >= 0.05)).sum())
print(f"regressors with t-2>t-1 significant and t-1>t not: {n_ok} of 8")
print("pooled medians of -log10 p by lag:", {lag: round(float(np.median(P[lag])), 3) for lag in LAGS})

**Result: reproduced.** The t-2 distribution lies above the t-1 distribution (weaker encoding) with p <
0.05 for all 8 regressors (pooled D = 0.207, p = 2.6e-29), while t-1 does not lie above t for 7 of 8
(risk is the exception, p = 0.0002; pooled p = 0.968).

## figs3a-risk-all-trials

> The risk regressor over all trials explains slightly more HFA variance than risk restricted to
> gamble trials, so risk is encoded whatever the subsequent choice.

In [ ]:
e_risk = SC[("risk", 0)]["encoding"]
all_prof = SC[("risk", 0)]["r2"][0][:, e_risk].mean(1) * 100
g_prof = SC[("risk_gamble_only", 0)]["r2"][0][:, e_risk].mean(1) * 100
print(f"risk-encoding electrodes (all-trial selection): {int(e_risk.sum())}")
print(f"game epoch peak mean %EV: all trials {all_prof.max():.2f} at {c[all_prof.argmax()]} ms; gamble trials only {g_prof.max():.2f} at {c[g_prof.argmax()]} ms")
post = c >= 100
print(f"mean over windows centred >= 100 ms: all {all_prof[post].mean():.2f}, gamble only {g_prof[post].mean():.2f}")
e2 = SC[("risk_gamble_only", 0)]["encoding"]
print(f"variant, electrodes selected on gamble trials ({int(e2.sum())}): peak gamble-only {SC[('risk_gamble_only', 0)]['r2'][0][:, e2].mean(1).max() * 100:.2f},"
      f" all-trial on the same electrodes {SC[('risk', 0)]['r2'][0][:, e2].mean(1).max() * 100:.2f}")
print("gamble-only trials have fewer trials, so the chance %EV is higher there: chance peak, all",
      f"{(SC[('risk', 0)]['perm_r2_enc'][:, 0].sum(0) / SC[('risk', 0)]['perm_count'].sum()).max() * 100:.2f},",
      f"gamble only {(SC[('risk_gamble_only', 0)]['perm_r2_enc'][:, 0].sum(0) / SC[('risk_gamble_only', 0)]['perm_count'].sum()).max() * 100:.2f}")

**Result: not reproduced.** Over the 84 risk-encoding electrodes the gamble-only regressor gives the
higher peak mean %EV (3.12 against 2.32 for all trials). Gamble trials are fewer, and the chance peak is
higher for them too (2.30 against 1.25), so the comparison of raw %EV favours the smaller set.

## figs3b-no-motor-encoding

> The direction of the response (left or right option) is not reflected in OFC HFA.

In [ ]:
s3b = summary(("left", 0))
print(s3b)
for ep in saez.EPOCHS:
    prof, ch = profile([("left", 0)], ep)
    print(f"{ep}: peak mean %EV {prof.max():.2f} at {c[prof.argmax()]} ms (chance profile max {ch.max():.2f})")
print("encoding > 95th percentile of chance:", s3b["encoding"] > s3b["chance 95th"],
      f"| percentile of the observed count among shuffles: {stats.percentileofscore(SC[('left', 0)]['perm_count'], s3b['encoding']):.1f}")

**Result: reproduced.** 31 electrodes (16.2%) meet the encoding rule for the left/right response,
below the mean of its shuffles (34.7) and the 95th percentile (49); the observed count is at the 34.5th
percentile of the shuffled counts.

## tables2-patients-coverage

> All signals were present in at least 6 of the 10 patients (mean 7.41 plus or minus 0.4).

In [ ]:
cov = pd.DataFrame([summary((k, lag)) for lag in (0, 1) for k in saez.CURRENT])[["regressor", "lag", "encoding", "patients"]]
print(cov.to_string(index=False))
grp_pat = {}
for gname, keys in GROUPS.items():
    any_enc = np.any([SC[k]["encoding"] for k in keys], 0)
    grp_pat[gname] = len(set(SUBJ[any_enc]))
ra = cov.patients.values
rb = cov[~((cov.regressor == "risk") & (cov.lag == 1))].patients.values
rc = np.array(list(grp_pat.values()))
for name, v in (("(a) 16 regressors", ra), ("(b) 15 without past risk", rb), ("(c) 4 groups", rc)):
    print(f"{name}: mean {v.mean():.2f} (SEM {stats.sem(v):.2f}), min {v.min()}, all >= 6: {bool((v >= 6).all())}")
print("groups:", grp_pat)
print("expected 7.41 +/- 0.4 (7.01 to 7.81), all >= 6")

**Result: not reproduced.** Every regressor is present in at least 6 patients (past risk in 6, the
others in 7 to 10), but the mean is 8.62 patients over the 16 regressors, 8.80 without past risk and 10
over the four groups, all above 7.41 +/- 0.4 and above the 7.75 that Table S2 gives counted the same way.

## fig2-example-electrodes

> Example electrodes show risk encoding with an inverted U relation between win probability and HFA
> at peak encoding, and regret encoding with a linear drop of HFA with regret.

In [ ]:
def at_peak(key, e_global):
    """HFA at the electrode's peak %EV window (over both epochs) for all its trials."""
    sc = SC[key]
    s = SUBJ[e_global]
    e_local = int(e_global - np.where(SUBJ == s)[0][0])
    ei, wi = np.unravel_index(sc["r2"][:, :, e_global].argmax(), (2, saez.N_WIN))
    return s, H[s][saez.EPOCHS[ei]][:, wi, e_local].astype(float)


quad, lin, quad_neg_any = [], [], []
for e in np.where(SC[("risk", 0)]["encoding"])[0]:
    s, y = at_peak(("risk", 0), e)
    p = REG[("winprob", 0)][s]
    X = np.c_[np.ones(len(p)), p, p ** 2]
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (res @ res) / (len(y) - 3))
    tq = beta[2] / se[2]
    quad.append((beta[2], 2 * stats.t.sf(abs(tq), len(y) - 3)))
for e in np.where(SC[("regret", 0)]["encoding"])[0]:
    s, y = at_peak(("regret", 0), e)
    lr = stats.linregress(REG[("regret", 0)][s], y)
    lin.append((lr.slope, lr.pvalue))
quad, lin = np.array(quad), np.array(lin)
print(f"risk-encoding electrodes {len(quad)}: negative quadratic term with p < 0.05 in {int(((quad[:, 0] < 0) & (quad[:, 1] < 0.05)).sum())},"
      f" positive with p < 0.05 in {int(((quad[:, 0] > 0) & (quad[:, 1] < 0.05)).sum())}")
print(f"regret-encoding electrodes {len(lin)}: significant linear slope in {int((lin[:, 1] < 0.05).sum())}"
      f" ({int(((lin[:, 0] > 0) & (lin[:, 1] < 0.05)).sum())} with HFA rising with regret, i.e. falling as regret becomes more negative;"
      f" {int(((lin[:, 0] < 0) & (lin[:, 1] < 0.05)).sum())} the other way)")

**Result: reproduced.** Of the 84 risk-encoding electrodes, 44 show a significant negative quadratic
term in win probability at their peak window (an inverted U) and 40 a significant positive one. All 64
regret-encoding electrodes show a significant linear slope at their peak window, 38 with HFA rising with
regret (falling as regret becomes more negative) and 26 the other way.

## behavior-past-no-effect

> None of the past trial characteristics significantly affected risk taking (all p > 0.15, random
> effects logistic regression).

All non-timeout rounds after the first; t-1 regressors from the rounds with the authors' convention
(0 for choice and outcome regressors after a timeout).

In [ ]:
rows = []
for s in saez.SUBJECTS:
    r = beh[s]["rounds"]
    idx = np.where(~r.timeout.values)[0]
    p1 = saez.past_regressors(r, idx, 1)
    d = pd.DataFrame({"subject": s, "gamble": r.gamble_ind.values[idx], "winprob": r.winprob.values[idx]})
    rows.append(pd.concat([d, p1.add_prefix("prev_")], axis=1))
BP = pd.concat(rows, ignore_index=True).dropna()
print("rounds used:", len(BP))
out = []
for k in saez.CURRENT:
    xk = BP["prev_" + k].values
    fa = saez.glmm_logit(BP.gamble.values, np.c_[np.ones(len(BP)), BP.winprob.values, xk], BP.subject.values)
    fb = saez.glmm_logit(BP.gamble.values, np.c_[np.ones(len(BP)), xk], BP.subject.values)
    out.append({"past regressor": saez.LABELS[k], "(a) beta": fa.estimate[2], "(a) p": fa.p[2], "(b) beta": fb.estimate[1], "(b) p": fb.p[1],
                "beta per SD (a)": fa.estimate[2] * xk.std()})
bpe = pd.DataFrame(out)
print(bpe.round(4).to_string(index=False))
print("reading (a): any p < 0.05:", bool((bpe["(a) p"] < 0.05).any()), "| min p", round(bpe["(a) p"].min(), 4),
      "| all p > 0.15:", bool((bpe["(a) p"] > 0.15).all()))
print("reading (b): any p < 0.05:", bool((bpe["(b) p"] < 0.05).any()), "| min p", round(bpe["(b) p"].min(), 4))

**Result: not reproduced.** With current win probability as a covariate (reading (a)), past risk
(p = 0.015) and past chosen value (p = 0.039) affect gamble choice at p < 0.05; RPE is at p = 0.047.
Alone (reading (b)), past risk (p = 0.0005) and past chosen value (p = 0.034) are significant. The
effects are small (about 0.1 log-odds per SD).

## figs6b-stepwise-similar

> Stepwise regression with model selection gave proportions of encoding electrodes per regressor
> similar to those of the 5-consecutive-window selection.

STAR Methods, Stepwise regression: per electrode and regressor, the sum of F over the longest
stretch of consecutive windows with p < 0.05; 10,000 label shuffles give a permutation p,
Bonferroni corrected over the electrodes ("across n = 192 electrodes"; the 191 analysed here are
used); surviving regressors enter forward selection by ANOVA at p < 0.05, starting from the one with
the highest peak %EV, adding the regressor with the largest residual improvement. Two details are
open and are assumptions: the stretch is taken in each epoch and the larger sum used; the model
comparison is made on the HFA of the window where the base regressor has its peak %EV (reading
(a)). Reading (b) shows the proportions after the permutation step alone, before model selection.

In [ ]:
# check of the procedure on a synthetic electrode that encodes x strongly, and one that does not
rng = np.random.default_rng(4999)
xs = rng.normal(size=150)
fake = {e: rng.normal(size=(150, saez.N_WIN, 2)).astype(np.float32) for e in saez.EPOCHS}
fake["game"][:, 25:40, 0] += xs[:, None]
print("synthetic check, permutation p (encoding, null electrode):", saez.stepwise_sumF(fake, xs, 2000, rng)[1].round(4))
t0 = time.time()
KEYS16 = [(k, 0) for k in saez.CURRENT] + [(k, 1) for k in saez.CURRENT]
perm_p = {}
for i, key in enumerate(KEYS16):
    rng = np.random.default_rng(5000 + i)
    perm_p[key] = np.concatenate([saez.stepwise_sumF(H[s], np.asarray(REG[key][s], float), 10000, rng)[1] for s in saez.SUBJECTS])
print(f"permutations done in {time.time() - t0:.0f} s")
bonf = saez.ALPHA / N_ELEC
survive = {k: perm_p[k] < bonf for k in KEYS16}
selected = {k: np.zeros(N_ELEC, bool) for k in KEYS16}
offset = 0
for s in saez.SUBJECTS:
    ne = H[s]["game"].shape[2]
    for e in range(ne):
        eg = offset + e
        cands = [k for k in KEYS16 if survive[k][eg]]
        if not cands:
            continue
        peak = {k: SC[k]["r2"][:, :, eg].max() for k in cands}
        base = max(cands, key=peak.get)
        ei, wi = np.unravel_index(SC[base]["r2"][:, :, eg].argmax(), (2, saez.N_WIN))
        y = H[s][saez.EPOCHS[ei]][:, wi, e].astype(float)
        Xd = pd.DataFrame({f"{k[0]}|{k[1]}": REG[k][s] for k in cands})
        ok = ~Xd.isna().any(axis=1).values
        chosen = saez.forward_select(y[ok], Xd[ok].reset_index(drop=True), f"{base[0]}|{base[1]}")
        for name in chosen:
            k0, l0 = name.split("|")
            selected[(k0, int(l0))][eg] = True
    offset += ne
sw = pd.DataFrame({"regressor": [saez.LABELS[k[0]] for k in KEYS16], "lag": [k[1] for k in KEYS16],
                   "5-window %": [100 * SC[k]["encoding"].mean() for k in KEYS16],
                   "stepwise %": [100 * selected[k].mean() for k in KEYS16],
                   "permutation step only %": [100 * survive[k].mean() for k in KEYS16]})
sw["difference"] = sw["stepwise %"] - sw["5-window %"]
print(sw.round(1).to_string(index=False))
for col in ("stepwise %", "permutation step only %"):
    rho = stats.spearmanr(sw["5-window %"], sw[col])
    print(f"{col}: (a) Spearman rho {rho.statistic:.2f}, p {rho.pvalue:.3f}; (b) max |difference| {np.abs(sw[col] - sw['5-window %']).max():.1f} points")
print("electrodes with any regressor after stepwise:", int(np.any(list(selected.values()), 0).sum()), "of", N_ELEC)

**Result: partially reproduced, resting on assumptions about the stepwise procedure.** The
procedure behaves as intended on a synthetic electrode (p = 0.0005, the smallest possible with 2,000
shuffles). On the data, few electrodes survive the Bonferroni step: 16 of 191 keep any regressor, and
the stepwise proportions are 0 to 2.1% against 20.9 to 49.7% for the 5-window selection. They are
ranked alike (Spearman rho 0.77, p < 0.001, reading (a) holds) but differ by up to 48.2 points (reading
(b) fails).

## fig7f-multiplexing

> Most electrodes (132 of 192, 68.7%) encoded at least two of the four signal types (choice,
> outcome, past choice, past outcome) and 30 of 192 (15.6%) encoded all four.

In [ ]:
def group_sets(sel):
    return {g: np.any([sel[k] for k in keys], 0) for g, keys in GROUPS.items()}


for name, sel in (("stepwise", selected), ("5-window (variant)", {k: SC[k]["encoding"] for k in KEYS16})):
    gs = group_sets(sel)
    ntypes = np.sum(list(gs.values()), 0)
    print(f"{name}: per group {dict((g, int(v.sum())) for g, v in gs.items())}; >= 2 types {int((ntypes >= 2).sum())} ({100 * (ntypes >= 2).mean():.1f}%),"
          f" all 4 {int((ntypes == 4).sum())} ({100 * (ntypes == 4).mean():.1f}%), any {int((ntypes >= 1).sum())} of {N_ELEC}")
print("expected: >= 2 types 132 +/- 13.2, all four 30 +/- 3")

**Result: not reproduced, resting on the stepwise assumptions.** Under the stepwise selection no
electrode encodes two or more types (16 encode one). Under the 5-window selection (variant) 166
electrodes (86.9%) encode two or more and 65 (34.0%) all four, both above 132 +/- 13.2 and 30 +/- 3.

## figs7-present-past-overlap

> Electrodes encoding both present and t-1 information were overrepresented, for choice (39 of 120,
> 32.5%, against an expected 31, p < 0.05) and for outcome (43 of 120, 35.8%, against an expected 34,
> p < 0.01), chi-square tests; many electrodes still encoded only present or only past information.

In [ ]:
def overlap(sets, base_mask, label):
    for cls in ("choice", "outcome"):
        a, b = sets[cls][base_mask], sets["past " + cls][base_mask]
        tab = np.array([[np.sum(a & b), np.sum(a & ~b)], [np.sum(~a & b), np.sum(~a & ~b)]])
        n = tab.sum()
        exp = a.sum() * b.sum() / n
        if (tab.sum(0) == 0).any() or (tab.sum(1) == 0).any():
            print(f"  {label} {cls}: N {n}, both {tab[0, 0]}, expected {exp:.1f}; a row or column is empty, no test")
            continue
        chi = stats.chi2_contingency(tab, correction=False)
        chiy = stats.chi2_contingency(tab, correction=True)
        print(f"  {label} {cls}: N {n}, both {tab[0, 0]} ({100 * tab[0, 0] / n:.1f}%), expected {exp:.1f}; present only {tab[0, 1]}, past only {tab[1, 0]};"
              f" chi2 {chi.statistic:.2f} p {chi.pvalue:.3g} (Yates p {chiy.pvalue:.3g}); overrepresented {tab[0, 0] > exp}")


gs_sw, gs_5 = group_sets(selected), group_sets({k: SC[k]["encoding"] for k in KEYS16})
any_sw = np.any(list(gs_sw.values()), 0)
any_5 = np.any(list(gs_5.values()), 0)
overlap(gs_sw, any_sw, "(a) stepwise, electrodes encoding any group")
overlap(gs_5, any_5, "(b) 5-window, electrodes encoding any group")
overlap(gs_sw, np.ones(N_ELEC, bool), "(c) stepwise, all electrodes")
overlap(gs_5, np.ones(N_ELEC, bool), "(c) 5-window, all electrodes")
print("expected: choice both 39 +/- 3.9 (p < 0.05), outcome both 43 +/- 4.3 (p < 0.01), each above the count expected under independence")

**Result: not reproduced, resting on the stepwise assumptions.** Under the stepwise selection the
sets are too small to test (both counts 0). Under the 5-window selection the counts of electrodes
encoding both present and past information are 99 (choice) and 101 (outcome) of 184, against 39 and 43
of 120; outcome is overrepresented (101 against 95.7 expected, p = 0.047; 101 against 92.2 over all
191, p = 0.002) but choice is not (99 against 99.9 expected, p = 0.73).

## mixed-effects-robust

> In nested mixed-effects models with patient and electrode as random effects, all regressors were
> significant (all p < 10^-4, Bonferroni corrected).

Reading (b) of the time point: HFA averaged over the windows that start at or after the event
(centres 100 to 1,900 ms), in each epoch, fitted with patient and electrode within patient as random
intercepts (statsmodels MixedLM, REML); Bonferroni over 16 regressors and 2 epochs, a regressor
counted when significant in either. Regressors one at a time (a) and all 16 together (b);
electrodes all (a) and encoding only (b, one model per regressor). Reading (a) of the time point (one
model per window) is run with the electrode as the only random intercept, as the nested model takes
about 20 s per fit and 1,824 fits do not fit the time limit; this is a deviation.

In [ ]:
POST = c >= 100


def long_frame(epoch, windows, electrodes=None):
    rows = []
    off = 0
    for s in saez.SUBJECTS:
        h = H[s][epoch]
        y = h[:, windows, :].mean(1) if np.ndim(windows) else h[:, windows, :]
        for e in range(h.shape[2]):
            if electrodes is None or electrodes[off + e]:
                d = {"y": y[:, e].astype(float), "subj": s, "elec": f"{s}_{e}"}
                for k in KEYS16:
                    d[f"{k[0]}_{k[1]}"] = REG[k][s]
                rows.append(pd.DataFrame(d))
        off += h.shape[2]
    return pd.concat(rows, ignore_index=True)


def fit_nested(args):
    kind, epoch, key = args
    if kind == "all 16, all":
        df = long_frame(epoch, POST)
        formula, keys = "y ~ " + " + ".join(names), names
    else:
        nm = f"{key[0]}_{key[1]}"
        df = long_frame(epoch, POST, SC[key]["encoding"] if kind == "one, encoding" else None)
        formula, keys = f"y ~ {nm}", [nm]
    df = df.dropna()
    m = smf.mixedlm(formula, df, groups="subj", re_formula="1", vc_formula={"elec": "0+C(elec)"}).fit(reml=True)
    return [{"model": kind, "epoch": epoch, "regressor": n, "beta": m.params[n], "p": m.pvalues[n]} for n in keys]


def fit_window(args):
    epoch, w = args
    df = long_frame(epoch, w)
    out = []
    for nm in names:
        d = df[["y", "elec", nm]].dropna()
        m = smf.mixedlm(f"y ~ {nm}", d, groups="elec").fit(reml=True)
        out.append({"epoch": epoch, "window": w, "regressor": nm, "beta": m.params[nm], "p": m.pvalues[nm]})
    return out


names = [f"{k[0]}_{k[1]}" for k in KEYS16]
POOL = lambda: ProcessPoolExecutor(3, mp_context=multiprocessing.get_context("fork"))
t0 = time.time()
jobs = [(kind, epoch, k) for epoch in saez.EPOCHS for k in KEYS16 for kind in ("one, all", "one, encoding")]
jobs += [("all 16, all", epoch, None) for epoch in saez.EPOCHS]
with POOL() as ex:
    mm = pd.DataFrame([row for rows in ex.map(fit_nested, jobs) for row in rows])
print(f"nested fits: {len(jobs)} in {time.time() - t0:.0f} s")
mm["p_bonf"] = np.minimum(mm.p * 32, 1)
best = mm.groupby(["model", "regressor"]).p_bonf.min().unstack(0)
print(best.map(lambda v: f"{v:.1e}").to_string())
for col in best.columns:
    print(f"{col}: regressors with corrected p < 1e-4: {int((best[col] < 1e-4).sum())} of 16; with corrected p < 0.05: {int((best[col] < 0.05).sum())}")
sign = mm[mm.model == "one, all"].pivot(index="regressor", columns="epoch", values="beta")
print("fixed-effect slopes, one regressor, all electrodes:\n", sign.round(3).to_string())

Reading (a) of the time point: one model per window (both epochs, 114 windows), each regressor
alone, all electrodes, electrode random intercept; Bonferroni over 16 regressors and 114 windows.

In [ ]:
t0 = time.time()
with POOL() as ex:
    pw = pd.DataFrame([row for rows in ex.map(fit_window, [(e, w) for e in saez.EPOCHS for w in range(saez.N_WIN)]) for row in rows])
pw["p_bonf"] = np.minimum(pw.p * 16 * 114, 1)
bw = pw.groupby("regressor").p_bonf.min()
print(f"per-window fits: {len(pw)} in {time.time() - t0:.0f} s")
print(bw.map(lambda v: f"{v:.1e}").to_string())
print(f"regressors with corrected p < 1e-4 in some window: {int((bw < 1e-4).sum())} of 16")

**Result: not reproduced.** No reading makes every regressor significant at corrected p < 10^-4. The
best is reading (a) of the time point (one model per window, electrode random intercept): 14 of 16, with
past chosen value (corrected p = 6.2e-03) and past risk (3.0e-03) short of it. With HFA averaged after
the event and the nested model (reading (b)), 4 of 16 (all electrodes) and 6 of 16 (encoding
electrodes) reach it; the model with all 16 regressors is rank-deficient (win and loss index add up to
gamble), and 2 of 16 reach it. Fixed-effect slopes average over electrodes whose encoding goes in either
direction, which the paper's %EV measure ignores.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| methods-electrode-trial-counts | partially reproduced |
| behavior-risk-neutral | reproduced |
| fig1c-choice-win-probability | reproduced |
| figs2a-choice-gamble-ev | reproduced |
| fig1c-patients-like-controls | blocked, data_missing |
| figs2b-rt-conflict | not reproduced |
| regressors-past-current-uncorrelated | not reproduced |
| figs6a-regressor-collinearity | reproduced |
| fig3-present-encoding | not reproduced |
| fig3-latency-duration-similar | reproduced (assumption) |
| hfa-more-than-fp | blocked, data_missing |
| figs4b-power-at-buttonpress | blocked, data_missing |
| fig4-past-encoding | partially reproduced |
| fig5-past-sustained | partially reproduced |
| behavior-past-no-effect | not reproduced |
| fig6-past-short-lived | reproduced |
| figs5d-sharp-boundary | reproduced |
| mixed-effects-robust | not reproduced |
| figs6b-stepwise-similar | partially reproduced (assumption) |
| figs3a-risk-all-trials | not reproduced |
| figs3b-no-motor-encoding | reproduced |
| anatomy-brodmann-counts | blocked, data_missing |
| fig7f-multiplexing | not reproduced (assumption) |
| figs7-present-past-overlap | not reproduced (assumption) |
| fig7-all-areas-similar | blocked, data_missing |
| no-anatomical-gradient | blocked, data_missing |
| tables2-patients-coverage | not reproduced |
| fig2-example-electrodes | reproduced |

**How the analysis differs from the paper's.** The deposited HFA is used as it is: already
window-averaged and, in the game epoch, baseline-corrected. Eight electrodes of s03 that copy
others are dropped (191 analysed against the paper's 192). The buttonpress epoch is read as locked to
the outcome reveal, as the data indicate. The regressors are the authors' own file; t-2 to t-5 follow
its convention. Random-effects logits are fitted by maximum likelihood with Gauss-Hermite quadrature.
The stepwise procedure takes the longest stretch in each epoch and compares models at the base
regressor's peak window (assumptions). The per-window mixed models use an electrode random intercept
only, for time.

**What was found about the data set.** The files hold 199 electrodes, of which seven in s03 are exact
copies of others and one (s03 electrode 42) holds copies of two different electrodes in its 1-kHz and
window arrays; without the seven, the per-patient counts match Table S2. The "buttonpress" epoch is
centred 470 to 600 ms after the csv's button press, and the gamble outcome is encoded from about 200 ms
after its time 0: it behaves as locked to the outcome reveal, not the button press. `reveal.time` is
31 to 47 ms after the button press in most rounds, not 550 ms. HFA trials carry no round number and
are fewer than the non-timeout rounds (1,753 against 1,839), without documentation of the exclusions;
they can be matched to rounds through the authors' regressor file. s10 played 138 rounds with reaction
times up to 80 s and no timeouts; timeouts are 5.1% of rounds, not 9.98%. The column `choice.time`,
the HFA units and the baseline subtraction are not described. The data set holds no broadband signal,
no electrode locations and no data from the healthy participants.